In [ ]:
from litellm import completion
from dotenv import load_dotenv
import os

from pydantic import BaseModel, ValidationError, RootModel
import pdfplumber
import json

from pydantic import BaseModel, Field, ConfigDict


class PPC(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    Informações_Gerais: str = Field(alias="1 Informações Gerais")
    Apresentação: str = Field(alias="2 Apresentação")
    Exposição_de_Motivos: str = Field(alias="3 Exposição de Motivos")
    Objetivos: str = Field(alias="4 Objetivos")
    Princípios_Norteadores_para_a_Formação_Profissional: str = Field(
        alias="5 Princípios Norteadores para a Formação Profissional"
    )
    Expectativas_da_Formação_Profissional: str = Field(
        alias="6 Expectativas da Formação Profissional"
    )
    Estrutura_Curricular: str = Field(alias="7 Estrutura Curricular")
    Estágio_Curricular: str = Field(alias="8 Estágio Curricular")
    Trabalho_de_Conclusão_de_Curso: str = Field(
        alias="9 Trabalho de Conclusão de Curso"
    )
    Atividades_Complementares: str = Field(alias="10 Atividades Complementares")
    Integração_Ensino_Pesquisa_e_Extensão: str = Field(
        alias="11 Integração Ensino, Pesquisa e Extensão"
    )
    Avaliação_do_Processo_de_Ensino_Aprendizagem: str = Field(
        alias="12 Avaliação do Processo de Ensino-Aprendizagem"
    )
    Avaliação_do_Projeto_de_Curso: str = Field(
        alias="13 Avaliação do Projeto de Curso"
    )
    Qualificação_de_Docentes_e_Técnico_Administrativos: str = Field(
        alias="14 Qualificação de Docentes e Técnico-Administrativos"
    )
    Requisitos_Legais_e_Normativos: str = Field(
        alias="15 Requisitos Legais e Normativos"
    )
    Dinâmica_das_Atividades_EAD: str = Field(alias="16 Dinâmica das Atividades (EAD)")
    Referências: str = Field(alias="17 Referências")
    Apêndices: str = Field(alias="18 Apêndices")
    Extras: str = Field(alias="Extras")

load_dotenv()
#MODEL = "ollama/qwen2.5:7b-instruct"
MODEL = "openrouter/nvidia/nemotron-3.5-lightning:free"
CAPITULOS_PROMPT = """
Você é um leitor de Projetos Pedagógicos de Curso e vai dividir o texto em capítulos.
NÃO MODIFIQUE OS TEXTOS.
Identifique quais capítulos estão nele e então seu output deve ser SOMENTE um JSON. 
Utilize SOMENTE os capítulos da lista abaixo:
{
  "Informações Gerais": "",
  "Apresentação": "",
  "Exposição de Motivos": "",
  "Objetivos": "",
  "Princípios Norteadores para a Formação Profissional": "",
  "Expectativas da Formação Profissional": "",
  "Estrutura Curricular": "",
  "Estágio Curricular": "",
  "Trabalho de Conclusão de Curso": "",
  "Atividades Complementares": "",
  "Integração Ensino, Pesquisa e Extensão": "",
  "Avaliação do Processo de Ensino-Aprendizagem": "",
  "Avaliação do Projeto de Curso": "",
  "Qualificação de Docentes e Técnico-Administrativos": "",
  "Requisitos Legais e Normativos": "",
  "Dinâmica das Atividades (EAD)": "",
  "Referências": "",
  "Apêndices": "",
  "Extras": ""
}


Você não lerá todas as páginas, então é possível que falte algo antes ou depois. Caso aconteça: coloque [falta] aonde isso possa ter acontecido.
Retorne SOMENTE o JSON.
"""

schema = PPC.model_json_schema()


def generate_response(messages, tools=None):
    response = completion(
        model=MODEL,
        messages=messages,
        tools=tools,
        
        temperature=0,
        response_format={
            "type": "json_object"
        },

    )
    return response

def resposta_valida(resultado_texto, chaves_esperadas):
    try:
        dados = json.loads(resultado_texto)
    except json.JSONDecodeError:
        return None
    if set(dados.keys()) == chaves_esperadas:
        return dados
    return None

def tamanho_pdf(arquivo):
    with pdfplumber.open(arquivo) as pdf:
        tamanho = len(pdf.pages)
    return tamanho

def ler_texto_pdf(arquivo, min, max):
    texto = ""
    with pdfplumber.open(arquivo) as pdf:
        for pagina in pdf.pages[min:max]:
            texto += (pagina.extract_text() or "") + "\n"
    return texto


def limpar_json(texto):
    texto = texto.strip()
    if texto.startswith("```json"):
        texto = texto.replace("```json", "", 1)
    if texto.endswith("```"):
        texto = texto[:-3]
    return texto.strip()


def separar_capitulos(pdf):
    tamanho = tamanho_pdf(pdf)
    d = {}
    pagina = 0
    while pagina <= 8:
        texto = ler_texto_pdf(pdf, pagina, pagina+10)
        messages = [
            {
                "role": "system",
                "content": CAPITULOS_PROMPT
            },
            {
                "role": "user",
                "content": texto
            }
        ]
        pagina += 8
        for tentativa in range(3):
            try:
                response = generate_response(messages)
                print(response) #debug
                resultado = response.choices[0].message.content
                resultado = limpar_json(resultado)
                print(resultado) #debug

                #ppc = PPC.model_validate_json(resultado)

                if resposta_valida(resultado, CAPITULOS_PROMPT):
                    d[pagina] = json.loads(resultado)
                    print(d[pagina]) #debug
                    break
            except Exception as e:
                print(f"Tentativa {tentativa+1}: {e}")
    return d


print(separar_capitulos('Cópia de Proposta de PPC - Matemática.pdf'))


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



16:11:14 - LiteLLM:WARNING: core_helpers.py:139 - Unmapped finish_reason 'error', defaulting to 'stop'


ModelResponse(id='gen-1786561335-Ut3z7uOw7CThDGohhtKV', created=1786561335, model='nvidia/nemotron-3.5-lightning:free', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content=None, role='assistant', tool_calls=None, function_call=None, reasoning_content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User is acting as a "reader of Course Pedagogical Projects" and wants to divide text into chapters.\n   - Instruction: "NÃO MODIFIQUE OS TEXTOS." (Do NOT modify the texts.)\n   - Identify which chapters are in the text.\n   - Output must be SOMENTE a JSON (only JSON).\n   - Use ONLY the chapters from the provided list.\n   - If some content is missing due to not reading all pages, put "[falta]" where that might have happened.\n   - Return ONLY the JSON.\n\n2.  **Identify the Chapters in the Provided Text:**\n   The text is from "ANEXO DA RESOLUÇÃO CONSEPEC/UFCAT Nº XX/2025 - PROJETO PEDAGÓGICO DO CURSO DE

16:21:16 - LiteLLM:WARNING: core_helpers.py:139 - Unmapped finish_reason 'error', defaulting to 'stop'


ModelResponse(id='gen-1786561935-hb5JRPWfm2c3SuI9LLVS', created=1786561935, model='nvidia/nemotron-3.5-lightning:free', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content=None, role='assistant', tool_calls=None, function_call=None, reasoning_content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User is acting as a "reader of Pedagogical Course Projects" (Projeto Pedagógico de Curso).\n   - Task: \nNow I need to wait for the output to stabilize and then provide the text into chapters.\n   - Constraint: Do NOT modify the texts.\n   - Identify which chapters are in the text.\n   - Output must be ONLY a JSON.\n   - Use ONLY the chapters from the provided list.\n   - If something is missing (not in the text), put "[falta]" where it might have happened.\n   - Return ONLY the JSON.\n   - The text is a pedagogical project course document (PROJETO PEDAGÓGICO DO CURSO DE GRADUAÇÃO EM MATEMÁTICA).\n   - Th

KeyboardInterrupt: 

: 